In [1]:
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
# import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        # Convert bytes to string
        data_str = l.decode('utf-8')
        # Replace 'false' with 'False' and 'true' with 'True'
        data_str = data_str.replace('false', 'False').replace('true', 'True')

        # Parse the JSON string into a dictionary
        yield eval(data_str)
'''
Set seeds
'''
seed = 999
random.seed(seed)
np.random.seed(seed)

In [2]:
DATA_PATH = '../data/'

In [3]:
DATASET = 'sports'

In [4]:
test_samples = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'negative_samples.txt'))
len(test_samples)

35598

In [5]:
sequential_data = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'sequential_data.txt'))
item_count = defaultdict(int)
user_items = defaultdict()

for line in sequential_data:
    user, items = line.strip().split(' ', 1)
    items = items.split(' ')
    items = [str(item) for item in items]
    user_items[user] = items
    for item in items:
        item_count[item] += 1

In [6]:
user_items['1']

['1', '2', '3', '4', '5', '6', '7', '8']

In [7]:
all_item = list(item_count.keys())

In [8]:
all_item[:4]

['1', '2', '3', '4']

In [9]:
datamaps = load_json(os.path.join(DATA_PATH, DATASET, 'datamaps.json'))
user2id = datamaps['user2id']
item2id = datamaps['item2id']
user_list = list(datamaps['user2id'].keys())
item_list = list(datamaps['item2id'].keys())
id2item = datamaps['id2item']
id2user = datamaps['id2user']

In [10]:
list(id2user.keys())[:4]

['1', '2', '3', '4']

In [11]:
print("#samples:",len(test_samples[0].split(' ',1)[1].split(' ')))
test_samples[0].split(' ',1)[1].split(' ')[:10]

#samples: 99


['4404',
 '10734',
 '4470',
 '5576',
 '12378',
 '1172',
 '3301',
 '7718',
 '11592',
 '13871']

In [12]:
train_negative = []
for user in tqdm(list(id2user.keys())):
    user_seq = user_items[user][:]
    user_seq = set([str(x) for x in user_seq])
    candidate_samples = []
    candidate_num = len(user_seq)
    already_samples = test_samples[int(user)-1].split(' ', 1)[1].split(' ')
    already_samples = set([str(x) for x in already_samples])
    while len(candidate_samples) < candidate_num:
        choices = [item for item in all_item if item not in user_seq]
        sample_ids = np.random.choice(choices, candidate_num, replace=False)
        sample_ids = [str(item) for item in sample_ids if (item not in user_seq)]
        sample_ids = [str(item) for item in sample_ids if (item not in already_samples)]
        sample_ids = [str(item) for item in sample_ids if (item not in candidate_samples)]
        candidate_samples.extend(sample_ids)
    candidate_samples = candidate_samples[:candidate_num]
    train_negative.append([user] + candidate_samples)
    

100%|██████████| 35598/35598 [01:32<00:00, 385.78it/s]


In [13]:
train_negative[0], user_items['1'], test_samples[0]

(['1', '17058', '1490', '3171', '887', '15984', '16790', '7594', '3671'],
 ['1', '2', '3', '4', '5', '6', '7', '8'],
 '1 4404 10734 4470 5576 12378 1172 3301 7718 11592 13871 7034 16069 3798 4311 17804 10894 11053 17934 13847 10996 8258 14965 16361 11835 16725 5829 9368 967 6939 5495 17369 14865 553 13945 8665 17496 17950 17424 6874 4956 215 6723 17938 1513 5948 14312 7570 18181 7421 2612 15200 7140 17158 6942 1761 1431 6377 3158 11149 10715 7759 3161 13139 8804 2716 16999 5918 16883 1458 2170 11431 14545 4822 11912 191 16920 3688 15645 501 3095 1192 13325 5512 6187 992 11847 15766 1054 3078 4967 13216 5456 11531 13031 3718 3511 11973 10177 17720')

In [14]:
len(train_negative)

35598

# Check overlaps

In [15]:
test_negs = []
for idx in range(len(test_samples)):
    lst = test_samples[idx].split(' ')[1:]
    user = test_samples[idx].split(' ')[0]
    for val in lst:
        test_negs.append((int(user) - 1, int(val)))
len(test_negs)

3524202

In [16]:
test_negs[:10]

[(0, 4404),
 (0, 10734),
 (0, 4470),
 (0, 5576),
 (0, 12378),
 (0, 1172),
 (0, 3301),
 (0, 7718),
 (0, 11592),
 (0, 13871)]

In [17]:
train_negs = []
for idx in range(len(train_negative)):
    lst = train_negative[idx]
    user = lst[0]
    items = lst[1:]
    for item in items:
        train_negs.append((int(user)-1,int(item)))

In [18]:
len(train_negs)

296337

In [19]:
train_negs[:10]

[(0, 17058),
 (0, 1490),
 (0, 3171),
 (0, 887),
 (0, 15984),
 (0, 16790),
 (0, 7594),
 (0, 3671),
 (1, 12918),
 (1, 10846)]

In [20]:
inter = []
for user,items in user_items.items():
    new_user = int(user) - 1
    for item in items:
        item = int(item)
        inter.append((new_user,item))

In [21]:
len(inter)

296337

In [22]:
inter[:10]

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (0, 7),
 (0, 8),
 (1, 9),
 (1, 10)]

In [23]:
len(set(train_negs))

296337

In [24]:
len(set(test_negs))

3524202

In [25]:
len(set(inter))

296337

In [26]:
common1 = set(train_negs).intersection(set(test_negs))
len(common1)

0

In [27]:
common2 = set(train_negs).intersection(set(inter))
len(common2)

0

In [28]:
common2

set()

In [29]:
save_pickle(train_negative,os.path.join(DATA_PATH,DATASET,'train-negatives.pkl'))